# Disease Prediction System

This notebook implements a symptom-based disease prediction system using a Decision Tree Classifier.

In [5]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import ipywidgets as widgets
from IPython.display import display, clear_output

## 1. Data Generation
Creating a synthetic dataset for 10 common diseases.

In [6]:
# Define diseases and their typical symptoms
diseases_data = {
    'Flu': ['fever', 'cough', 'fatigue', 'sore_throat', 'body_aches'],
    'Common Cold': ['sneezing', 'runny_nose', 'sore_throat', 'cough'],
    'Malaria': ['fever', 'chills', 'sweating', 'headache', 'nausea', 'muscle_pain'],
    'Typhoid': ['high_fever', 'weakness', 'stomach_pain', 'headache', 'rash'],
    'Dengue': ['high_fever', 'rash', 'joint_pain', 'vomiting', 'eye_pain'],
    'Migraine': ['severe_headache', 'nausea', 'sensitivity_to_light', 'dizziness'],
    'Diabetes': ['excessive_thirst', 'frequent_urination', 'fatigue', 'blurred_vision'],
    'Hypertension': ['headache', 'shortness_of_breath', 'nosebleed', 'dizziness'],
    'Pneumonia': ['cough_with_phlegm', 'fever', 'chills', 'difficulty_breathing'],
    'Covid-19': ['fever', 'dry_cough', 'loss_of_taste_smell', 'difficulty_breathing']
}

# Get all unique symptoms
all_symptoms = set()
for symptoms in diseases_data.values():
    all_symptoms.update(symptoms)
all_symptoms = sorted(list(all_symptoms))

# Generate synthetic samples
num_samples_per_disease = 50
data = []

for disease, symptoms in diseases_data.items():
    for _ in range(num_samples_per_disease):
        sample = {symptom: 0 for symptom in all_symptoms}
        # Set symptoms present for this disease to 1
        for s in symptoms:
            # Add some noise: most symptoms are present, a few might be missing
            if np.random.random() > 0.1: # 90% chance to have the symptom
                 sample[s] = 1
        
        # Add some random noise (symptoms not typical for the disease)
        for s in all_symptoms:
            if s not in symptoms and np.random.random() > 0.98: # 2% chance of random symptom
                sample[s] = 1
        
        sample['Disease'] = disease
        data.append(sample)

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (500, 32)


,blurred_vision,body_aches,chills,cough,cough_with_phlegm,difficulty_breathing,dizziness,dry_cough,excessive_thirst,eye_pain,...,sensitivity_to_light,severe_headache,shortness_of_breath,sneezing,sore_throat,stomach_pain,sweating,vomiting,weakness,Disease
0,0,1,0,1,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,0,Flu
1,0,1,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,Flu
2,0,1,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,Flu
3,0,1,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,Flu
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,Flu


## 2. Preprocessing and Training

In [7]:
X = df[all_symptoms]
y = df['Disease']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# Evaluation
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 89.00%


## 3. User Interface
Interactive symptom checker.

In [ ]:
# UI Components
style = {'description_width': 'initial'}

header = widgets.HTML("<h2>Disease Prediction System</h2>")

name_input = widgets.Text(
    description='Patient Name:',
    placeholder='Enter patient name',
    style=style,
    layout=widgets.Layout(width='50%')
)

symptom_label = widgets.HTML("<h3>Select Symptoms:</h3>")

symptom_checkboxes = [widgets.Checkbox(value=False, description=s, indent=False) for s in all_symptoms]
grid = widgets.GridBox(symptom_checkboxes, layout=widgets.Layout(grid_template_columns="repeat(3, 300px)"))

predict_btn = widgets.Button(
    description="Predict Disease",
    button_style='success', 
    icon='check'
)

reset_btn = widgets.Button(
    description="Reset",
    button_style='warning',
    icon='refresh'
)

output_area = widgets.Output()

def on_predict_clicked(b):
    with output_area:
        clear_output()
        name = name_input.value.strip()
        if not name:
            print("Please enter the patient's name.")
            return

        # Get selected symptoms
        selected_symptoms = [box.description for box in symptom_checkboxes if box.value]
        
        if not selected_symptoms:
            print("Please select at least one symptom.")
            return
            
        # Prepare input vector
        input_vector = pd.DataFrame([np.zeros(len(all_symptoms))], columns=all_symptoms)
        for s in selected_symptoms:
            input_vector[s] = 1
            
        # Predict
        prediction = clf.predict(input_vector)[0]
        probabilities = clf.predict_proba(input_vector)[0]
        confidence = max(probabilities) * 100
        
        display(widgets.HTML(f"""
        <div style="border: 2px solid #4CAF50; padding: 10px; border-radius: 5px; background-color: #f9f9f9;">
            <h4>Report for: {name}</h4>
            <p><b>Predicted Disease:</b> <span style="color: red; font-size: 16px;">{prediction}</span></p>
            <p><b>Confidence:</b> {confidence:.2f}%</p>
            <p><b>Symptoms Reported:</b> {', '.join(selected_symptoms)}</p>
        </div>
        """))

def on_reset_clicked(b):
    name_input.value = ''
    for box in symptom_checkboxes:
        box.value = False
    with output_area:
        clear_output()

predict_btn.on_click(on_predict_clicked)
reset_btn.on_click(on_reset_clicked)

# Layout
buttons_box = widgets.HBox([predict_btn, reset_btn], layout=widgets.Layout(margin='20px 0px 0px 0px'))
ui_container = widgets.VBox([
    header,
    name_input,
    symptom_label,
    grid,
    buttons_box,
    widgets.HTML("<hr>"),
    output_area
])

display(ui_container)